In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

# Augmetations

- Noise
- Volume
- Pitch Shifting
- Speed (however this must not slow things down to mean that they are longer than max seconds)

In [ ]:
import torch
import torchaudio


from IPython.display import Audio

# from pprint import pprint
# download example
# torch.hub.download_url_to_file('https://models.silero.ai/vad_models/en.wav', 'en_example.wav')

model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad", model="silero_vad", force_reload=True
)

(get_speech_timestamps, _, read_audio, *_) = utils

sampling_rate = 16000  # also accepts 8000

# get speech timestamps from full audio file

In [ ]:
from nanodrz.data import gather_speakers_from_folder, artificial_diarisation_sample
from nanodrz import data
from nanodrz.utils import play, visualise_annotation

In [ ]:
speakers = data.librilight_small()
print(speakers)
sr = 16000

audio = torchaudio.load(speakers[0].utts[0][1])[0]
speech_timestamps = get_speech_timestamps(audio, model, sampling_rate=sampling_rate)
print(speech_timestamps)
# visualise_annotation([[s["start"]/16000, s["end"]/16000,"A"] for s in speech_timestamps])

for utt in speakers[2].utts:
    audio = torchaudio.load(utt[1])[0]
    speech_timestamps = get_speech_timestamps(audio, model, sampling_rate=sampling_rate)
    if speech_timestamps[0]["start"] > 1600 and speech_timestamps[-1]["end"] < (audio.shape[-1] - 1600):
        play(audio)
        print(speech_timestamps)
        visualise_annotation([[s["start"]/16000, s["end"]/16000,"A"] for s in speech_timestamps])


In [ ]:
audio, labels = data.artificial_diarisation_sample(
    list(speakers),
    min_secs=15,
    max_secs=20,
    num_speakers=3,
    silence_max=2,
    interrupt_max=2,
)
print(audio.shape[-1] / 16000)
visualise_annotation(labels)
play(audio, 16000)
speech_timestamps = get_speech_timestamps(audio, model, sampling_rate=sampling_rate)
print(speech_timestamps)
visualise_annotation([[s["start"]/16000, s["end"]/16000,"A"] for s in speech_timestamps])

In [ ]:
from nanodrz import augmentations as augs

augment = augs.build_augmentations(
    [
        (augs.SinVol(sr), 1),
        (augs.AddNoise(), 1),
    ]
)

In [ ]:
import torch

with torch.inference_mode():
    augment(audio)

In [ ]:
from denoiser import pretrained
import torch
import torchaudio

torch.cuda.set_device("cuda:1")
denoiser =  pretrained.dns64().cpu().eval()

@torch.inference_mode()
def denoise(audio_file, sr=None):
    global denoiser
    if type(audio_file) is str: 
        audio, sr = torchaudio.load(audio_file)
    else:
        audio = audio_file
        assert sr is not None, "You must provide sample rate for loaded audio"
    
    audio = audio.sum(dim=0, keepdim=True)
    # audio = resample(sr, denoiser.sample_rate, audio)
    B = 40
    denoiser = denoiser.cuda()
    wav = audio.split(B*denoiser.sample_rate, dim=1)
    denoised = []
    for w in wav:
        denoised.append(denoiser(w.cuda()))
    denoiser = denoiser.cpu()
    denoised = torch.cat(denoised, dim=-1)
    return denoised

In [ ]:
from torchaudio.transforms import Vad
import torchaudio
from nanodrz.utils import contains_non_silence

for utt in speakers[1].utts[:10]:
    utt = utt[1]
    play(utt)
    audio,sr = torchaudio.load(utt)
    out = denoise(audio, sr)
    
    play(out)
    print(sr, audio.shape[-1]/sr, out.shape[-1]/sr, out.shape)